# Minimal repro: Unsloth DPO + Gemma 4 vision hangs at tokenization

**Symptom:** `DPOTrainer.train()` hangs silently at `Tokenizing 0/52 [00:00<?, ? examples/s]` with both `dataset_num_proc=1` and `dataset_num_proc=2`. CPU at ~0%, GPU at 0%, no error, no progress.

**Setup:** Gemma 4 E4B 4-bit, vision layers unfrozen, LoRA r=16, fresh SFT-warmstart adapter (or fresh LoRA — either reproduces). Synthetic preference pairs: 1024x1024 RGB images + short prompts/chosen/rejected.

**Diagnostics that work (function-level):**
- `processor(images=..., text=...)` direct call: returns in 0.1s
- `_UnslothDPOTrainer.process_row(features=pairs[0], ...)` direct call: returns in 0s
- Manual Python loop calling `process_row` over all pairs: returns in ~2s

**What hangs:** `dataset.map(process_row)` inside `DPOTrainer._prepare_dataset` — the same function works fine outside `dataset.map`.

**Workaround attempts that didn't help:**
- Setting `HF_DATASETS_CACHE` to a known-writable path on persistent volume
- Switching `dataset_num_proc` between 1 and 2
- Using `Features({"images": [HFImage()], ...})` for explicit Arrow type
- Pre-tokenizing manually and passing tokenized dataset (TRL still calls `dataset.map(..., remove_columns=["chosen","rejected"])` and errors because those columns are gone)

**Environment:** Modal A100 (40GB), Linux, Python 3.12.


## 1. Install

In [2]:

%uv pip install pillow==11.3.0
%uv pip install --upgrade transformers
%uv pip install trl
%uv pip install "peft==0.19.1"
%uv pip install "unsloth==2026.4.7"


Using Python 3.12.6 environment at: /usr/local
Audited 1 package in 4ms
Note: you may need to restart the kernel to use updated packages.
Using Python 3.12.6 environment at: /usr/local
Resolved 27 packages in 63ms
Prepared 1 package in 0.63ms
Uninstalled 1 package in 1ms
Installed 1 package in 53ms
 - fsspec==2026.2.0
 + fsspec==2026.3.0
Note: you may need to restart the kernel to use updated packages.
Using Python 3.12.6 environment at: /usr/local
Resolved 72 packages in 55ms
Uninstalled 1 package in 0.98ms
Installed 1 package in 54ms
 - fsspec==2026.3.0
 + fsspec==2026.2.0
Note: you may need to restart the kernel to use updated packages.
Using Python 3.12.6 environment at: /usr/local
Audited 1 package in 9ms
Note: you may need to restart the kernel to use updated packages.
Using Python 3.12.6 environment at: /usr/local
Resolved 100 packages in 204ms
⠙ Preparing packages... (0/35)
⠙ Preparing packages... (0/35)
⠙ Preparing packages... (0/35)
typeguard  ------------------------------  

In [3]:
# Pin versions for the bug report
import importlib.metadata as md
for pkg in ["unsloth", "trl", "transformers", "datasets", "peft", "torch", "pillow"]:
    try:
        print(f"{pkg}: {md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"{pkg}: NOT INSTALLED")

unsloth: 2026.4.7
trl: 0.24.0
transformers: 5.5.0
datasets: 4.3.0
peft: 0.19.1
torch: 2.10.0
pillow: 11.3.0


## 2. Imports

In [4]:
from unsloth import FastVisionModel       # MUST be first
from trl import DPOTrainer, DPOConfig
from datasets import Dataset
from PIL import Image
import torch
import time
import os

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


## 3. Load Gemma 4 E4B 4-bit + fresh LoRA adapter

Use Unsloth's `FastVisionModel.get_peft_model` (NOT `PeftModel.from_pretrained`, which fails on `Gemma4ClippableLinear` module resolution).

In [5]:
model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/gemma-4-e4b-it",      
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)
print(f"Base loaded. GPU: {torch.cuda.memory_allocated()/1e9:.1f} GB")

model = FastVisionModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    finetune_vision_layers=True,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    target_modules="all-linear",
)
print("Trainable params:")
model.print_trainable_parameters()

==((====))==  Unsloth 2026.4.7: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100 80GB PCIe. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

Base loaded. GPU: 10.9 GB
Trainable params:
trainable params: 41,222,144 || all params: 8,037,378,592 || trainable%: 0.5129


## 4. Synthetic preference pairs (52 examples)

Just blank 1024x1024 RGB PIL images + short text. Real images and real preferences are not needed to reproduce — the hang is structural, not data-dependent.

Schema: flat `{images: [PIL], prompt: str, chosen: str, rejected: str}` — this is what Unsloth's vision DPO `process_row` expects (it does NOT want chat-template content blocks).

In [6]:
def make_pair(idx: int):
    img = Image.new("RGB", (1024, 1024), color=(idx % 256, 128, 200))
    return {
        "images": [img],
        "prompt": "Describe the image in one sentence.",
        "chosen": f"Image {idx} shows a solid color block.",
        "rejected": f"Image {idx} contains nothing useful at all.",
    }

pairs = [make_pair(i) for i in range(52)]
print(f"Built {len(pairs)} synthetic pairs")
print(f"Sample image: {pairs[0]['images'][0].size}")

Built 52 synthetic pairs
Sample image: (1024, 1024)


## 5. Diagnostic: confirm `process_row` works on its own

This call IS the function that `dataset.map` will invoke 52 times during DPOTrainer prep. It works fine in isolation.

In [7]:
from unsloth_compiled_cache.UnslothDPOTrainer import _UnslothDPOTrainer

start = time.time()
out = _UnslothDPOTrainer.process_row(
    features=pairs[0],
    processing_class=tokenizer,
    max_prompt_length=1024,
    max_completion_length=None,
    add_special_tokens=False,
)
print(f"Single call OK in {time.time()-start:.2f}s. Keys: {list(out.keys())}")

Single call OK in 0.03s. Keys: ['prompt_input_ids', 'pixel_values', 'chosen_input_ids', 'rejected_input_ids']


In [8]:
# Manual loop: call process_row on every pair. Should finish in ~2s.
start = time.time()
processed = []
for p in pairs:
    out = _UnslothDPOTrainer.process_row(
        features=p,
        processing_class=tokenizer,
        max_prompt_length=1024,
        max_completion_length=None,
        add_special_tokens=False,
    )
    processed.append(out)
print(f"Manual loop over {len(pairs)} pairs: {time.time()-start:.2f}s")

Manual loop over 52 pairs: 0.65s


## 6. Repro: DPOTrainer hangs at tokenization step

Wraps `pairs` in a `Dataset` and instantiates DPOTrainer. The trainer's `_prepare_dataset` calls `dataset.map(process_row, ...)` internally — that's where it hangs.

Expected behaviour: tokenization completes in seconds (since `process_row` is fast).

Actual behaviour: progress bar sits at `0/52 [00:00<?, ? examples/s]`, CPU drops to ~0%, GPU at 0%, no progress, no error. Have to interrupt the kernel.

In [ ]:
pairs_ds = Dataset.from_list(pairs)
print(f"Dataset: {len(pairs_ds)} examples, columns: {pairs_ds.column_names}")

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,
    processing_class=tokenizer,
    train_dataset=pairs_ds,
    args=DPOConfig(
        beta=0.1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        learning_rate=5e-6,
        output_dir="/tmp/dpo-repro",
        max_length=2048,
        max_prompt_length=1024,
        remove_unused_columns=False,
        logging_steps=1,
        dataset_num_proc=1,    # also tried 2 — both hang
    ),
)

# Hangs here ↓
dpo_trainer.train()

Dataset: 52 examples, columns: ['images', 'prompt', 'chosen', 'rejected']


Extracting prompt in train dataset (num_proc=1):   0%|          | 0/52 [00:00<?, ? examples/s]

Applying chat template to train dataset (num_proc=1):   0%|          | 0/52 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=1):   0%|          | 0/52 [00:00<?, ? examples/s]

## What I've tried that didn't fix it

1. **`dataset_num_proc=1` and `=2`** — both hang. Single-process means no fork issue, but still wedged.
2. **Forced `HF_DATASETS_CACHE` to a known-writable persistent path** — no change.
3. **Explicit `Features({"images": [HFImage()], "prompt": Value("string"), ...})` schema** — Dataset constructs in 0s, but trainer still hangs at the same step.
4. **Pre-tokenize via manual loop** then pass tokenized dataset (with `prompt_input_ids`, `pixel_values`, `chosen_input_ids`, `rejected_input_ids` columns) — TRL's `_prepare_dataset` still calls `dataset.map(..., remove_columns=["chosen", "rejected"])` which then errors because those columns are gone.

## Question

- Is there a known issue with Unsloth's compiled `_UnslothDPOTrainer` + Gemma 4 vision + `dataset.map` on this stack?
- Is there a way to instruct DPOTrainer to skip its data prep step entirely when a pre-tokenized dataset is passed in?
- Or a config flag to disable the multiprocess path inside `_prepare_dataset`?